# Transforming with Large Language Models: Translation, Formatting, Tone & Proofreading

A major capability of Large Language Models (LLMs) is **transformation**: taking an input in one representation, style, or structure and converting it into another. 

Where developers previously relied on brittle regular expressions, distinct translation APIs, and separate spelling/grammar toolchains, an LLM handles these tasks flexibly through targeted prompting.

### Core Transformation Workflows Covered:
1. **Language Translation & Identification:** Single-language translation, multi-lingual broadcasting, register/formality handling, and building a universal IT customer-support translator.
2. **Tone & Style Adaptation:** Transforming casual text/slang into formal corporate correspondence.
3. **Cross-Format Conversion:** Ingesting structured raw data (JSON/Python dictionaries) and generating valid, renderable HTML tables.
4. **Grammar Correction & Proofreading:** Catching subject-verb disagreement, homonyms, and typos, along with visualizing changes using diff packages (`redlines`).
5. **Style-Guide Adaptation:** Rewriting informal user reviews to conform to formal academic/publication standards (e.g., APA style).

## 1. Setup & Multi-Provider Client Configuration

Load secrets securely from your local `.env` file.
- Supports standard OpenAI API credentials (`OPENAI_API_KEY`).
- Supports Google Gemini via the OpenAI-compatible endpoint (`GEMINI_API_KEY` directed to `https://generativelanguage.googleapis.com/v1beta/openai/`).

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

# 1. Load API key from .env file
_ = load_dotenv(find_dotenv())

# 2. Initialize the OpenAI client pointing to the working Gemini endpoint
client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

### Helper Function: `get_completion`
Encapsulates chat completion requests. We default `temperature=0` to preserve deterministic output for strict conversions and translations.

In [2]:
# # 3. helper function to make it easier to get completions from the model
# def get_completion(prompt, model="gemini-3.8-flash"):
#     messages = [{"role": "user", "content": prompt}]
#     response = client.chat.completions.create(
#         model=model,
#         messages=messages,
#         temperature=0,  # 0 for deterministic output
#     )
#     return response.choices[0].message.content

# 3. Define prioritized models (from fastest/latest to reliable backups)
from openai import OpenAI, APIError, RateLimitError, InternalServerError, NotFoundError
DEFAULT_MODELS = [
    "gemini-flash-latest",
    "gemini-pro-latest",
    "gemini-flash-lite-latest",
    "gemini-2.0-flash-lite",
    "gemini-3.8-flash",
    "gemini-2.5-flash",
    "gemini-2.0-flash",
    "gemini-1.5-flash",
    "gemini-1.5-pro",
]

def get_completion(prompt, models=DEFAULT_MODELS, max_retries_per_model=2):
    """
    Attempts to get a completion by cascading through a prioritized list of models.
    Handles 429 (Quota/Rate Limits), 503 (High Demand/Unavailable), and 404 (Not Found).
    """
    # If a single model string is passed by mistake, wrap it in a list
    if isinstance(models, str):
        models = [models] + [m for m in DEFAULT_MODELS if m != models]

    messages = [{"role": "user", "content": prompt}]
    last_exception = None

    for model_name in models:
        delay = 1.0  # Initial delay for temporary errors

        for attempt in range(max_retries_per_model):
            try:
                response = client.chat.completions.create(
                    model=model_name,
                    messages=messages,
                    temperature=0,  # Deterministic output
                )
                return response.choices[0].message.content

            except RateLimitError as e:
                # 429 Quota Exceeded (Per-model limit reached) -> Switch immediately
                print(f"[429 Quota Exceeded] '{model_name}' limit reached. Switching to next model...")
                last_exception = e
                break  # Break inner loop to try next model immediately

            except NotFoundError as e:
                # 404 Model Not Found/Retired -> Switch immediately
                print(f"[404 Not Found] '{model_name}' is unavailable on this API version. Switching...")
                last_exception = e
                break

            except (InternalServerError, APIError) as e:
                # 503 / 500 Temporary Server Overload -> Retry once or switch
                last_exception = e
                status_code = getattr(e, "status_code", None)

                if attempt < max_retries_per_model - 1:
                    print(f"[{status_code or 'Error'}] '{model_name}' busy. Retrying in {delay}s...")
                    time.sleep(delay)
                    delay *= 2  # Exponential backoff
                else:
                    print(f"'{model_name}' failed after {max_retries_per_model} attempts. Switching to next model...")

    # If all models in the list fail:
    raise RuntimeError(f"All configured models failed. Last error: {last_exception}")

###### messages = [{"role": "user", "content": prompt}]: Formats your input prompt into the standard Chat Completion format, indicating that a human user is asking the question.

###### client.chat.completions.create(...): Sends the request to Gemini's server.

###### temperature=0: Sets the degree of randomness. A temperature of 0 ensures the model gives consistent, deterministic, and factual outputs rather than creative variations.

###### return response.choices[0].message.content: Extracts the generated text from the model's response and returns it as a string.

## 2. Language Translation & Identification

Because foundation LLMs are trained on massive multilingual web corpora, they possess intrinsic cross-lingual capabilities across hundreds of languages without requiring standalone translation software.

### 2.1 Basic Translation & Language Identification

In [3]:
# Translation into a target language
prompt_es = """
Translate the following English text to Spanish: 
```Hi, I would like to order a blender```
"""
print("--- Translation to Spanish ---")
print(get_completion(prompt_es))

# Identifying the source language of an unknown phrase
prompt_id = """
Tell me which language this is: 
```Combien coûte le lampadaire?```
"""
print("\n--- Language Identification ---")
print(get_completion(prompt_id))

--- Translation to Spanish ---
Here is the translation to Spanish:

"**Hola, me gustaría pedir una licuadora.**"

*(Note: In Spain, a blender is usually called a **batidora** instead of a **licuadora**).*

--- Language Identification ---
That sentence is in **French**. 

It translates to: *"How much does the floor lamp (or street lamp) cost?"*


### 2.2 Multi-Target Translation & Persona Emulation
An LLM can project the same source message into multiple cultural, linguistic, or fictional personas simultaneously.

In [ ]:
prompt_multi = """
Translate the following text to French, Spanish, and English Pirate: 
```I want to order a basketball```
"""
print("--- Multi-Target Translation ---")
print(get_completion(prompt_multi))

### 2.3 Handling Social Context: Formal vs. Informal Registers
Many languages (e.g., Spanish, German, French, Korean) distinguish grammatical forms based on the social hierarchy or familiarity between speaker and listener (e.g., *tú* vs. *usted* in Spanish).

In [ ]:
prompt_register = """
Translate the following text to Spanish in both the 
formal and informal forms: 
'Would you like to order a pillow?'
"""
print("--- Formal vs. Informal Translation ---")
print(get_completion(prompt_register))

## 3. Real-World Application: Automated Universal Support Translator

Imagine managing an enterprise IT helpdesk for a multinational corporation. Employees submit tickets in their respective native languages. 

Below, we loop over an array of tickets, identify each language, and translate it into standardized languages (English and Korean) for regional engineering dispatch.

In [ ]:
user_messages = [
    "La performance du système est plus lente que d'habitude.",  # French: System is slower than usual
    "Mi monitor tiene píxeles que no se iluminan.",              # Spanish: Monitor has dead pixels
    "Il mio mouse non funziona",                                 # Italian: Mouse not working
    "Mój klawisz Ctrl jest zepsuty",                             # Polish: Broken Ctrl key
    "我的屏幕在闪烁"                                               # Chinese: Screen is flickering
]

for idx, issue in enumerate(user_messages, start=1):
    # 1. Identify Language
    prompt_lang = f"Tell me what language this is in 1-2 words: ```{issue}```"
    lang = get_completion(prompt_lang).strip()
    
    # 2. Universal Translation
    prompt_trans = f"""
    Translate the following text into English and Korean. 
    Format output as:
    English: <english translation>
    Korean: <korean translation>
    
    Text: ```{issue}```
    """
    translation = get_completion(prompt_trans).strip()
    
    print(f"--- Ticket {idx} | Detected Language: {lang} ---")
    print(f"Original: {issue}")
    print(f"{translation}\n")

## 4. Tone Transformation: Adapting to Intended Audiences

Tone adaptation shifts phrasing to match communication channels—such as translating raw Slack/chat messages or colloquial slang into formal business correspondence.

In [ ]:
prompt_tone = """
Translate the following from casual slang to a formal business letter: 
'Dude, This is Joe, check out this spec on this standing lamp.'
"""
response_tone = get_completion(prompt_tone)
print("--- Slang to Formal Business Letter ---")
print(response_tone)

## 5. Cross-Format Conversion: JSON / Dict to HTML Table

LLMs excel at parsing machine data structures (JSON, CSV, Markdown, XML) and generating syntactically valid markup without manual templating.

In [ ]:
from IPython.display import display, HTML, Markdown

# Input structured Python dictionary
data_json = {
    "restaurant employees": [
        {"name": "Shyam", "email": "shyamjaiswal@gmail.com"},
        {"name": "Bob", "email": "bob32@gmail.com"},
        {"name": "Jai", "email": "jai87@gmail.com"}
    ]
}

prompt_format = f"""
Translate the following Python dictionary from JSON to an HTML 
table with column headers and title:
{data_json}
"""

response_html = get_completion(prompt_format)
print("--- Raw Generated HTML ---")
print(response_html)

# Render formatted HTML table directly in notebook
display(HTML(response_html))

## 6. Grammar & Spell Checking (Proofreading)

To instruct an LLM to proofread, explicitly instruct it to *'proofread'* or *'proofread and correct'*. This is particularly effective at resolving subtle errors that standard rule-based spellcheckers miss, such as homophones (*their* vs. *there* vs. *they're*, *affect* vs. *effect*) and subject-verb disagreements.

### 6.1 Batch Proofreading Challenging Sentences

In [ ]:
sentences = [ 
    "The girl with the black and white puppies have a ball.",               # Subject-verb agreement
    "Yolanda has her notebook.",                                           # Correct baseline
    "Its going to be a long day. Does the car need it’s oil changed?",     # Possessive vs. contraction
    "Their goes my freedom. There going to bring they’re suitcases.",       # Homonyms
    "Your going to need you’re notebook.",                                  # Homonyms
    "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Affect vs. effect
    "This phrase is to cherck chatGPT for speling abilitty"                 # Explicit spelling errors
]

for s in sentences:
    prompt_proof = f"""
    Proofread and correct the following text and rewrite the corrected version. 
    If you don't find any errors, just say "No errors found". 
    Don't use any punctuation or quotes around the output:
    ```{s}```
    """
    result = get_completion(prompt_proof)
    print(f"Original:  {s}")
    print(f"Corrected: {result.strip()}\n")

### 6.2 Visualizing Revisions Using Diffs (`redlines`)

When proofreading long-form text (such as product reviews), inspecting full text revisions can make it difficult to locate small changes. 

Using the `redlines` package, we compute an inline Markdown diff highlighting additions and deletions.

In [ ]:
# If redlines is not installed in your local environment, uncomment and run:
# !pip install redlines

review_text = """
Got this for my daughter for her birthday cuz she keeps taking 
mine from my room.  Yes, adults also like pandas too.  She takes 
it everywhere with her, and it's super soft and cute.  One of the 
ears is a bit lower than the other, and I don't think that was 
designed to be asymmetrical. It's a bit small for what I paid for it 
though. I think there might be other options that are bigger for 
the same price.  It arrived a day earlier than expected, so I got 
to play with it myself before I gave it to my daughter.
"""

prompt_review = f"Proofread and correct this review: ```{review_text}```"
corrected_review = get_completion(prompt_review)

try:
    from redlines import Redlines
    diff = Redlines(review_text, corrected_review)
    display(Markdown(diff.output_markdown))
except ImportError:
    print("--- Corrected Output (redlines not installed) ---")
    print(corrected_review)

## 7. Advanced Style & Publication Rewriting (APA Format in Markdown)

Transformations can encompass multiple complex dimensions simultaneously: correcting grammar, elevating vocabulary for an advanced audience, applying academic style conventions (e.g., APA), and outputting in structured Markdown.

In [ ]:
prompt_apa = f"""
Proofread and correct this review. Make it more compelling. 
Ensure it follows APA style guide and targets an advanced reader. 
Output in markdown format.

Text: ```{review_text}```
"""

response_apa = get_completion(prompt_apa)
display(Markdown(response_apa))

## 8. Summary & Transformation Takeaways

| Transformation Type | Common Use Cases | Key Prompting Tactics |
| :--- | :--- | :--- |
| **Language Translation** | Multilingual customer support, localization | Specify target language, formality register (*formal* vs. *informal*), and concise output limits. |
| **Tone Adaptation** | Converting drafts across business contexts | Define source tone and desired persona (e.g., *slang to formal corporate proposal*). |
| **Format Conversion** | Ingesting raw APIs into UI templates | Explicitly state source schema and target markup (e.g., *JSON/dict to HTML table with headers*). |
| **Proofreading** | Eliminating homophone errors and grammar slips | Instruct model to *'proofread and correct'*; pair with `redlines` to review inline diffs. |
| **Style Guide Compliance** | Publishing, corporate communications | State target guidelines (*APA style*, *technical reader*) and output structure (*Markdown*). |